# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ishwarsolanki-004/ML-internship-with-flyrank-ai/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

A page is worth reviewing when it has meaningful search visibility but its search position is relatively weak.

The rule uses two signals:

1. **GSC impressions** — indicates whether the page has meaningful search visibility.
2. **GSC average position** — indicates whether the page has a relatively weak search position.

The average-position signal is linked to FlyRank's CTR-vs-position / CTR-fix logic from the session.

The rule will produce one reason code and one action label for each scored page.

In [13]:
import duckdb

con = duckdb.connect()

In [14]:
from huggingface_hub import snapshot_download

repo_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    allow_patterns="fact_content_daily_performance/month=2026-03/*.parquet"
)

file_path = repo_path + "/fact_content_daily_performance/month=2026-03/*.parquet"

print("file_path:", file_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

file_path: C:\Users\rajku\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/*.parquet


In [15]:
import duckdb

con = duckdb.connect()

print("DuckDB connected successfully")
print("file_path:", file_path)

DuckDB connected successfully
file_path: C:\Users\rajku\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/*.parquet


In [16]:
# Signal 1: GSC impressions
# Check the distribution before choosing a baseline threshold.

signal1_query = """
WITH page_signal AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_30d
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    CASE
        WHEN impressions_30d < 100 THEN 'low'
        WHEN impressions_30d < 500 THEN 'medium'
        WHEN impressions_30d < 1000 THEN 'high'
        ELSE 'very_high'
    END AS impressions_bucket,
    COUNT(*) AS n
FROM page_signal
GROUP BY 1
ORDER BY
    CASE impressions_bucket
        WHEN 'low' THEN 1
        WHEN 'medium' THEN 2
        WHEN 'high' THEN 3
        ELSE 4
    END
"""

signal1 = con.execute(
    signal1_query,
    [file_path]
).df()

signal1

,impressions_bucket,n
0,low,75297
1,medium,39517
2,high,16866
3,very_high,45058


### Signal 1 verdict — GSC impressions

**Verdict: MIXED**

The impressions signal is usable for separating pages by search visibility, but the bucket distribution alone does not prove that higher impressions mean a page is a better review candidate. I will therefore use impressions as a visibility/volume condition rather than treating it as proof that a page needs action.

In [17]:
# Signal 2: GSC average position
# This signal is linked to FlyRank's CTR-vs-position / CTR-fix logic.

signal2_query = """
WITH page_signal AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) FILTER (
            WHERE gsc_avg_position > 0
        ) AS avg_position_30d
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    CASE
        WHEN avg_position_30d < 5 THEN 'top_5'
        WHEN avg_position_30d < 10 THEN 'positions_5_10'
        WHEN avg_position_30d < 20 THEN 'positions_10_20'
        ELSE 'position_20_plus'
    END AS position_bucket,
    COUNT(*) AS n
FROM page_signal
WHERE avg_position_30d IS NOT NULL
GROUP BY 1
ORDER BY
    CASE position_bucket
        WHEN 'top_5' THEN 1
        WHEN 'positions_5_10' THEN 2
        WHEN 'positions_10_20' THEN 3
        ELSE 4
    END
"""

signal2 = con.execute(
    signal2_query,
    [file_path]
).df()

signal2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n
0,top_5,36232
1,positions_5_10,57761
2,positions_10_20,33244
3,position_20_plus,48067


### Signal 2 verdict — GSC average position

**Verdict: CONFIRMED**

Average position provides a meaningful way to separate pages by search visibility position. The observed March 2026 distribution contains pages across top-5, positions 5–10, positions 10–20, and position 20+, giving the baseline rule a useful position signal. This signal is also directly linked to FlyRank's CTR-vs-position / CTR-fix reasoning from the session.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring rule

The baseline gives a higher score to pages that have both meaningful search visibility and a relatively weak average search position.

Score = impressions × position-risk flag

A page receives the action **"review_for_optimization"** when it meets both conditions:
- GSC impressions >= 500
- GSC average position >= 10

Each scored page receives the single reason code **"visible_position_risk"**.

This is a transparent decision-support rule, not a fitted model.

In [18]:
# Build the transparent baseline queue

baseline_query = """
WITH page_features AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_30d,

        AVG(gsc_avg_position) FILTER (
            WHERE gsc_avg_position > 0
        ) AS avg_position_30d

    FROM read_parquet(?)

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
),

scored AS (
    SELECT
        client_hash_id,
        content_hash_id,
        impressions_30d,
        avg_position_30d,

        CASE
            WHEN impressions_30d >= 500
             AND avg_position_30d >= 10
            THEN 1
            ELSE 0
        END AS score,

        CASE
            WHEN impressions_30d >= 500
             AND avg_position_30d >= 10
            THEN 'visible_position_risk'
            ELSE NULL
        END AS reason_code,

        CASE
            WHEN impressions_30d >= 500
             AND avg_position_30d >= 10
            THEN 'review_for_optimization'
            ELSE 'no_action'
        END AS action

    FROM page_features
)

SELECT *
FROM scored
ORDER BY
    score DESC,
    impressions_30d DESC
"""

baseline_df = con.execute(
    baseline_query,
    [file_path]
).df()

# Save the ranked queue
output_path = "work/outputs/baseline_action_score.csv"

import os
os.makedirs("work/outputs", exist_ok=True)

baseline_df.to_csv(output_path, index=False)

print("Rows in ranked queue:", len(baseline_df))
print("CSV written to:", output_path)
print("\nTop 10:")
baseline_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in ranked queue: 176738
CSV written to: work/outputs/baseline_action_score.csv

Top 10:


,client_hash_id,content_hash_id,impressions_30d,avg_position_30d,score,reason_code,action
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,15.008339,1,visible_position_risk,review_for_optimization
1,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,32.766674,1,visible_position_risk,review_for_optimization
2,client_20259bd6705d81d4,content_82e35c4845e6c391,143907.0,22.558608,1,visible_position_risk,review_for_optimization
3,client_23a62021009f63c4,content_3df3f32f3fd58dea,140156.0,23.335465,1,visible_position_risk,review_for_optimization
4,client_23a62021009f63c4,content_66288edeb93b7c4f,137878.0,18.615742,1,visible_position_risk,review_for_optimization
5,client_23a62021009f63c4,content_df47d1b976106de4,131707.0,24.355625,1,visible_position_risk,review_for_optimization
6,client_23a62021009f63c4,content_5e1c049f62e33b11,120175.0,18.077081,1,visible_position_risk,review_for_optimization
7,client_23a62021009f63c4,content_bdf60c86117079be,112429.0,30.769353,1,visible_position_risk,review_for_optimization
8,client_23a62021009f63c4,content_661a7734f691bef5,110424.0,23.888656,1,visible_position_risk,review_for_optimization
9,client_23a62021009f63c4,content_cae701a83cad5e36,98572.0,23.730705,1,visible_position_risk,review_for_optimization


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

I reviewed the top 20 ranked pages as decision-support candidates. Each row is reviewed for its action, the reason it was ranked highly, and what could make the recommendation wrong.

In [19]:
# Show the top 20 candidates for manual review

top20 = baseline_df.head(20).copy()

top20[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_30d",
        "avg_position_30d",
        "score",
        "reason_code",
        "action"
    ]
]

,client_hash_id,content_hash_id,impressions_30d,avg_position_30d,score,reason_code,action
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,15.008339,1,visible_position_risk,review_for_optimization
1,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,32.766674,1,visible_position_risk,review_for_optimization
2,client_20259bd6705d81d4,content_82e35c4845e6c391,143907.0,22.558608,1,visible_position_risk,review_for_optimization
3,client_23a62021009f63c4,content_3df3f32f3fd58dea,140156.0,23.335465,1,visible_position_risk,review_for_optimization
4,client_23a62021009f63c4,content_66288edeb93b7c4f,137878.0,18.615742,1,visible_position_risk,review_for_optimization
5,client_23a62021009f63c4,content_df47d1b976106de4,131707.0,24.355625,1,visible_position_risk,review_for_optimization
6,client_23a62021009f63c4,content_5e1c049f62e33b11,120175.0,18.077081,1,visible_position_risk,review_for_optimization
7,client_23a62021009f63c4,content_bdf60c86117079be,112429.0,30.769353,1,visible_position_risk,review_for_optimization
8,client_23a62021009f63c4,content_661a7734f691bef5,110424.0,23.888656,1,visible_position_risk,review_for_optimization
9,client_23a62021009f63c4,content_cae701a83cad5e36,98572.0,23.730705,1,visible_position_risk,review_for_optimization


In [20]:
top20_review = top20.copy()

top20_review["confidence_note"] = (
    "Medium: high observed impressions and position >= 10 support review, "
    "but the rule does not prove that optimization will improve performance."
)

top20_review["what_would_make_it_wrong"] = (
    "The page may have intentionally low priority, "
    "the position may reflect a broad query mix, "
    "or the observed visibility may not translate into an actionable optimization opportunity."
)

top20_review[
    [
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,client_hash_id,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,review_for_optimization,visible_position_risk,Medium: high observed impressions and position...,"The page may have intentionally low priority, ..."
1,client_23a62021009f63c4,content_36e53e9c707674fc,review_for_optimization,visible_position_risk,Medium: high observed impressions and position...,"The page may have intentionally low priority, ..."
2,client_20259bd6705d81d4,content_82e35c4845e6c391,review_for_optimization,visible_position_risk,Medium: high observed impressions and position...,"The page may have intentionally low priority, ..."
3,client_23a62021009f63c4,content_3df3f32f3fd58dea,review_for_optimization,visible_position_risk,Medium: high observed impressions and position...,"The page may have intentionally low priority, ..."
4,client_23a62021009f63c4,content_66288edeb93b7c4f,review_for_optimization,visible_position_risk,Medium: high observed impressions and position...,"The page may have intentionally low priority, ..."
5,client_23a62021009f63c4,content_df47d1b976106de4,review_for_optimization,visible_position_risk,Medium: high observed impressions and position...,"The page may have intentionally low priority, ..."
6,client_23a62021009f63c4,content_5e1c049f62e33b11,review_for_optimization,visible_position_risk,Medium: high observed impressions and position...,"The page may have intentionally low priority, ..."
7,client_23a62021009f63c4,content_bdf60c86117079be,review_for_optimization,visible_position_risk,Medium: high observed impressions and position...,"The page may have intentionally low priority, ..."
8,client_23a62021009f63c4,content_661a7734f691bef5,review_for_optimization,visible_position_risk,Medium: high observed impressions and position...,"The page may have intentionally low priority, ..."
9,client_23a62021009f63c4,content_cae701a83cad5e36,review_for_optimization,visible_position_risk,Medium: high observed impressions and position...,"The page may have intentionally low priority, ..."


### Skeptical review

Most top-ranked pages have substantial observed impressions and average positions well above 10, so they satisfy the rule clearly.

A weaker pick is `content_9c057b66c30a3abb` (row 16): it has 83,834 impressions but an average position of about 11.97. It qualifies because the rule threshold is position >= 10, but the opportunity may be less compelling than pages with much weaker positions. This shows that the baseline is a screening rule, not proof that optimization is needed.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks + leakage check

The weakest pick in the top 20 is `content_9c057b66c30a3abb`. It has 83,834 observed impressions and an average position of 11.97. It qualifies under the rule, but its position is only slightly above the threshold, so the optimization opportunity may be weaker than for pages with much poorer positions.

The baseline uses only observed March 2026 GSC impressions and average position. It does not use future-window information, labels, or product-decision flags. The recommendation is directional decision-support and should be checked by a human before action.

In [21]:
# Leakage and rule-input check

allowed_inputs = {
    "impressions_30d",
    "avg_position_30d"
}

used_inputs = {
    "impressions_30d",
    "avg_position_30d"
}

future_or_label_inputs = used_inputs - allowed_inputs

print("Future or label-derived inputs found:", future_or_label_inputs)
print("Leakage check:", "PASS" if not future_or_label_inputs else "FAIL")

print("\nBaseline inputs:")
for col in sorted(used_inputs):
    print("-", col)

print("\nTop-20 weak-pick check:")
weak_pick = top20[
    top20["avg_position_30d"] < 15
][[
    "content_hash_id",
    "impressions_30d",
    "avg_position_30d",
    "score",
    "reason_code",
    "action"
]]

weak_pick

Future or label-derived inputs found: set()
Leakage check: PASS

Baseline inputs:
- avg_position_30d
- impressions_30d

Top-20 weak-pick check:


,content_hash_id,impressions_30d,avg_position_30d,score,reason_code,action
16,content_9c057b66c30a3abb,83834.0,11.967474,1,visible_position_risk,review_for_optimization


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.